In [1]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from abc import ABC, abstractmethod
import glob
import time
import sys
import copy

# ==========================================
# 1. 数据加载与预处理类
# ==========================================
class HonorOfKingsDataLoader:
    def __init__(self, base_dir="data"):
        self.base_dir = base_dir
        self.name_to_id = {}
        self.id_to_name = {}
        self.base_win_rates = {}
        self.hero_relations = {} # 存储每个英雄的 synergy 和 counter 数据
        
    def load_base_mappings(self):
        # 加载 1.json (Name -> ID)
        with open(os.path.join(self.base_dir, "1.json"), "r", encoding="utf-8") as f:
            data_1 = json.load(f)
            for item in data_1:
                self.name_to_id[item["name"]] = str(item["id"])
                self.id_to_name[str(item["id"])] = item["name"]
                
        # 加载 20260529_top.csv (基础胜率)
        top_df = pd.read_csv(os.path.join(self.base_dir, "20260529_top.csv"))
        for _, row in top_df.iterrows():
            hero_name = row["英雄"].strip()
            # 转换为浮点数格式 (如 50.5% -> 0.505)
            wr_str = str(row["胜率"]).replace("%", "")
            self.base_win_rates[hero_name] = float(wr_str) / 100.0

    def load_hero_relations(self):
        # 遍历 data/heroes/{id}.json
        heroes_dir = os.path.join(self.base_dir, "heroes")
        if not os.path.exists(heroes_dir):
            print(f"Warning: {heroes_dir} 不存在，关系指数将默认为 0")
            return
            
        for file_name in os.listdir(heroes_dir):
            if file_name.endswith(".json"):
                hero_id = file_name.split(".")[0]
                hero_name = self.id_to_name.get(hero_id)
                if not hero_name:
                    continue
                
                with open(os.path.join(heroes_dir, file_name), "r", encoding="utf-8") as f:
                    detail = json.load(f)
                
                # 初始化该英雄的映射关系
                self.hero_relations[hero_name] = {
                    "synergy": {}, # 队友 -> (matches, index)
                    "counter": {}  # 对手 -> (matches, index)
                }
                
                # 队友正/负协同
                for item in detail.get("goodSynergies", []) + detail.get("badSynergies", []):
                    self.hero_relations[hero_name]["synergy"][item["heroName"]] = (
                        item.get("totalMatches", 0), item.get("synergyIndex", 0.0)
                    )
                # 对手克制/被克制
                for item in detail.get("counters", []) + detail.get("counteredBy", []):
                    self.hero_relations[hero_name]["counter"][item["heroName"]] = (
                        item.get("totalMatches", 0), item.get("advantageIndex", 0.0)
                    )

    def get_relation_features(self, hero_a, hero_b, relation_type="synergy"):
        """获取 hero_a 与 hero_b 组合的 (场次, 胜率变化)"""
        if hero_a in self.hero_relations and hero_b in self.hero_relations[hero_a][relation_type]:
            return self.hero_relations[hero_a][relation_type][hero_b]
        return (0, 0.0)


    def process_lineups(self):
        # 1. 动态扫描所有以 hero_battle_lineups 开头的 csv 文件
        search_pattern = os.path.join(self.base_dir, "hero_battle_lineups*.csv")
        csv_files = glob.glob(search_pattern)
        
        print(f" 📂 找到 {len(csv_files)} 个符合条件的阵容数据文件。")
        
        raw_data = []
        for csv_path in csv_files:
            # print(f" -> 正在读取: {os.path.basename(csv_path)}")
            with open(csv_path, "r", encoding="utf-8") as f:
                for line in f:
                    if "未集齐10人" in line:
                        continue
                    parts = line.strip().split(",")
                    if len(parts) < 14:  # 确保列数足够解析出 10 人阵容
                        continue
                    raw_data.append(parts)
                    
        processed_features = []
        seen_lineups = set()  # 用于阵容去重的集合
        duplicate_count = 0   # 统计去重数
        
        for row in raw_data:
            # 解析原始字段
            hero1, hero2, _, winner_tag = row[0], row[1], row[2], row[3]
            # 解析两个阵营的英雄列表（去除双引号）
            team_a_str = ",".join(row[4:9]).replace('"', '')
            team_b_str = ",".join(row[9:14]).replace('"', '')
            
            team_a = [h.strip() for h in team_a_str.split(",") if h.strip()]
            team_b = [h.strip() for h in team_b_str.split(",") if h.strip()]
            
            if len(team_a) != 5 or len(team_b) != 5:
                continue  # 双重保险，确保满10人
                
            # --------------------------------------------------------
            # ⚡ 镜像对抗去重新逻辑：不看人头顺序，不看红蓝阵营前后
            # --------------------------------------------------------
            # 队内排序，消除选人顺序带来的干扰
            sorted_a = tuple(sorted(team_a))
            sorted_b = tuple(sorted(team_b))
            
            # 强行令字典序小的队伍排在前面，消除“A对阵B”和“B对阵A”镜像重复干扰
            if sorted_a < sorted_b:
                lineup_key = (sorted_a, sorted_b)
            else:
                lineup_key = (sorted_b, sorted_a)
                
            if lineup_key in seen_lineups:
                duplicate_count += 1
                continue  # 触发去重条件，跳过此条数据
            seen_lineups.add(lineup_key)
            # --------------------------------------------------------
            
            # 判断原本是谁包含了 阵营1/2 的特征英雄
            real_team1 = team_a if hero1 in team_a else team_b
            real_team2 = team_b if hero1 in team_a else team_a
            
            # 根据胜负关系进行调整，使第一个阵营永远是获胜方
            if "阵营1 胜利" in winner_tag:
                win_team, lose_team = real_team1, real_team2
            else:
                win_team, lose_team = real_team2, real_team1
                
            # --- 构建 5 组核心特征 ---
            # 1 & 2. 双方各自英雄的固有胜率 (默认0.5)
            win_base_wr = [self.base_win_rates.get(h, 0.5) for h in win_team]
            lose_base_wr = [self.base_win_rates.get(h, 0.5) for h in lose_team]
            
            # 3. 获胜方英雄两两组合(Synergy) -> 5x5
            win_synergy = []
            for h1 in win_team:
                row_feat = []
                for h2 in win_team:
                    row_feat.append(self.get_relation_features(h1, h2, "synergy"))
                win_synergy.append(row_feat)
                
            # 4. 失败方英雄两两组合(Synergy) -> 5x5
            lose_synergy = []
            for h1 in lose_team:
                row_feat = []
                for h2 in lose_team:
                    row_feat.append(self.get_relation_features(h1, h2, "synergy"))
                lose_synergy.append(row_feat)
                
            # 5. 获胜方 vs 失败方英雄组合(Counter) -> 5x5
            win_vs_lose_counter = []
            for h1 in win_team:
                row_feat = []
                for h2 in lose_team:
                    row_feat.append(self.get_relation_features(h1, h2, "counter"))
                win_vs_lose_counter.append(row_feat)
                
            processed_features.append({
                "win_team": win_team,
                "lose_team": lose_team,
                "win_base_wr": win_base_wr,
                "lose_base_wr": lose_base_wr,
                "win_synergy": win_synergy,
                "lose_synergy": lose_synergy,
                "win_vs_lose_counter": win_vs_lose_counter
            })
            
        print(f" ✅ 多源数据加载完毕！总解析阵容: {len(raw_data)}，过滤完全重复阵容: {duplicate_count} 条，最终有效样本数: {len(processed_features)}")
        return processed_features

# ==========================================
# 3. 评估与主流程
# ==========================================
def evaluate_strategy(strategy, train_data, test_data):
    # 训练模型
    strategy.train(train_data)
    
    # 测试模型
    correct_predictions = 0
    for sample in test_data:
        pred = strategy.predict(sample)
        if pred == 1: # 因为测试集里全被我们调成了前队赢，所以 pred==1 代表预测正确
            correct_predictions += 1
            
    accuracy = correct_predictions / len(test_data) if test_data else 0
    return accuracy


# 初始化数据加载器
# 确保当前路径有 data 文件夹且放入了你的文件
loader = HonorOfKingsDataLoader(base_dir="data")

print("正在加载基础映射与胜率数据...")
loader.load_base_mappings()
print("正在加载英雄对抗克制关系...")
loader.load_hero_relations()

print("正在处理原始对局阵容...")
all_samples = loader.process_lineups()
print(f"有效对局总数: {len(all_samples)}")

if len(all_samples) == 0:
    print("未找到有效对局数据，请检查数据路径与格式。")
else:
    # 1:1 固定种子切分训练集与测试集
    train_data, test_data = train_test_split(all_samples, test_size=0.5, random_state=42)
    print(f"训练集大小: {len(train_data)}, 测试集大小: {len(test_data)}")

正在加载基础映射与胜率数据...
正在加载英雄对抗克制关系...
正在处理原始对局阵容...
 📂 找到 6 个符合条件的阵容数据文件。
 ✅ 多源数据加载完毕！总解析阵容: 54870，过滤完全重复阵容: 12467 条，最终有效样本数: 42403
有效对局总数: 42403
训练集大小: 21201, 测试集大小: 21202


In [14]:
# ==========================================
# 2. 预测策略抽象基类与基础实现
# ==========================================
class BasePredictionStrategy(ABC):
    
    def flip_sample(self, sample):
        original_counter = sample["win_vs_lose_counter"]
        flipped_counter = []
        
        for b_idx in range(5):
            row_feat = []
            for a_idx in range(5):
                # 提取原矩阵 [a_idx][b_idx] 的场次和指数
                matches, adv_idx = original_counter[a_idx][b_idx]
                # 转置过来后，指数取负号
                row_feat.append((matches, -adv_idx))
            flipped_counter.append(row_feat)
        
        flipped_sample = {
            "win_base_wr": sample["lose_base_wr"],      # 原后队变现前队
            "lose_base_wr": sample["win_base_wr"],       # 原前队变现后队
            "win_synergy": sample["lose_synergy"],       # 原后队协同变现前队协同
            "lose_synergy": sample["win_synergy"],       # 原前队协同变现后队协同
            "win_vs_lose_counter": flipped_counter       # 严格转置并取负后的克制矩阵
        }
        return flipped_sample
        
    def _flatten_features(self, sample):
        """将复杂的网格特征压平成一维向量作为模型输入"""
        feat_vector = []
        feat_vector.extend(sample["win_base_wr"])
        feat_vector.extend(sample["lose_base_wr"])
        # 展开 5x5 的 队友与对手关系值
        for i in range(5):
            for j in range(5):
                feat_vector.append(sample["win_synergy"][i][j][1])
                feat_vector.append(sample["lose_synergy"][i][j][1])
                feat_vector.append(sample["win_vs_lose_counter"][i][j][1])
        return feat_vector

    def train_data_aug(self, train_data):
        X_train, y_train = [], []
        for sample in train_data:
            # 正常顺序（前队赢，标签为1）
            X_train.append(self._flatten_features(sample))
            y_train.append(1)
            
            # 对称数据增强：将前后队调换（此时变成后队赢，标签为0）
            # 这能防止模型死记“前队永远赢”的规律
            X_train.append(self._flatten_features(self.flip_sample(sample)))
            y_train.append(0)

        return X_train, y_train
    
            
    def _get_smoothed_val(self, rel_tuple):
        """提取关系元组 (matches, index)，并进行贝叶斯平滑收缩"""
        matches, index_val = rel_tuple[0], rel_tuple[1]
        if matches == 0:
            return 0.0
        # 场次极低时，收缩系数接近0；场次高时，保持原值
        shrinkage = matches / (matches + self.min_matches)
        return index_val * shrinkage
    
    def _get_smoothed_vals_batch(self, rel_tuples_list):
        """
        内部辅助函数：矩阵化批量处理贝叶斯平滑
        输入维度: [Batch, 5, 5, 2] (最后一维包含 matches 和 index)
        """
        arr = np.array(rel_tuples_list, dtype=np.float32)
        matches = arr[..., 0]
        indices = arr[..., 1]
        
        # 贝叶斯平滑：当 min_matches 为 0 时不进行任何收缩
        if self.min_matches == 0:
            return indices
            
        mask = matches > 0
        shrinkage = np.where(mask, matches / (matches + self.min_matches), 0.0)
        return indices * shrinkage
    
    def _wr_to_logit(self, wr):
        """将基础胜率转换为对数几率"""
        wr = max(0.001, min(0.999, wr))
        return np.log(wr / (1.0 - wr))
    
    def _wr_to_logit_batch(self, wr_array):
        """
        ⚡ 矩阵化计算对数几率：Logit = ln(p / (1-p))
        带有 1e-5 的安全截断，防止胜率刚好为 0 或 1 时导致 log(0) 报错
        """
        wr_clipped = np.clip(wr_array, 1e-5, 1.0 - 1e-5)
        return np.log(wr_clipped / (1.0 - wr_clipped))
        
    @abstractmethod
    def train(self, train_data):
        """留给复杂机器学习模型的训练接口"""
        pass
        
    # @abstractmethod
    # def predict(self, sample):
    #     """
    #     输入单条样本特征，预测哪一边获胜。
    #     由于输入时已经将实际获胜方全部调整为前队，
    #     所以如果策略认为前队胜，返回 1 (预测正确)；认为后队胜，返回 0 (预测错误)。
    #     """
    #     pass

    @abstractmethod
    def predict_batch(self, samples):
        """
        输入一个样本列表 (Batch)，返回一个包含 0 或 1 的 list 或 numpy 数组。
        """
        return [self.predict(s) for s in samples]


In [9]:
# =====================================================================
# 核心策略：零参数、纯线性量纲对齐博弈模型
# =====================================================================

class OptimizedSimpleStrategy(BasePredictionStrategy):
    def __init__(self, w_base=100.0, alpha=1, beta=1, min_matches=0):
        """
        :param w_base: 基础胜率放大权重（将 0.53 映射为 53 分，与百分比指数对齐量纲）
        :param alpha: 队友协同分权重
        :param beta: 对手克制分权重
        :param min_matches: 贝叶斯平滑常数（过滤低频偶发脏数据）
        """
        self.w_base = w_base
        self.alpha = alpha
        self.beta = beta
        self.min_matches = min_matches

    def train(self, train_data):
        # 纯规则/超参数驱动，无需显式拟合参数
        pass

    def predict_batch(self, samples):
        """
        ⚡ 全矩阵化 Batch 预测，接收样本列表，直接返回 0/1 的 NumPy 数组
        """
        batch_size = len(samples)
        if batch_size == 0:
            return np.array([], dtype=np.int32)
        
        # 1. 基础胜率分量纲放大：提取所有样本的胜率列表并按行求和 -> [Batch]
        win_base = np.array([s["win_base_wr"] for s in samples], dtype=np.float32).sum(axis=1)
        lose_base = np.array([s["lose_base_wr"] for s in samples], dtype=np.float32).sum(axis=1)
        
        score_team1 = win_base * self.w_base
        score_team2 = lose_base * self.w_base
        
        # 2. 提取并平滑 5x5 的关系矩阵 -> 产出维度均为 [Batch, 5, 5]
        win_syn_smooth = self._get_smoothed_vals_batch([s["win_synergy"] for s in samples])
        lose_syn_smooth = self._get_smoothed_vals_batch([s["lose_synergy"] for s in samples])
        cnt_smooth = self._get_smoothed_vals_batch([s["win_vs_lose_counter"] for s in samples])
        
        # 3. 队友协同分博弈 (消除对角线 i == j 的干扰)
        eye_mask = np.eye(5, dtype=bool)
        win_syn_smooth[:, eye_mask] = 0.0
        lose_syn_smooth[:, eye_mask] = 0.0
        
        # 对 5x5 区域进行整体求和 -> [Batch]
        score_team1 += self.alpha * win_syn_smooth.sum(axis=(1, 2))
        score_team2 += self.alpha * lose_syn_smooth.sum(axis=(1, 2))
        
        # 4. 对手克制分博弈 (严格双向镜像对抗)
        cnt_sum = cnt_smooth.sum(axis=(1, 2))
        score_team1 += self.beta * cnt_sum
        score_team2 += self.beta * (-cnt_sum)
        
        # 哪边分数高，预测哪边获胜 (输出为 1 或 0)
        return np.where(score_team1 >= score_team2, 1, 0)

In [18]:
import time
import sys
import numpy as np
from itertools import product

def run_train_search_and_test_batch(strategy_class, param_grid, train_data, test_data):
    """
    通用矩阵加速版全流程评估脚本：支持任意策略类的传入与多维参数搜索
    
    :param strategy_class: 策略的类名，例如 OptimizedSimpleStrategy (注意不要加括号)
    :param param_grid: 字典格式的搜索网格，Key 必须与类的 __init__ 参数名完全一致
    :param train_data: 训练数据集
    :param test_data: 测试数据集
    """
    print("=" * 80)
    print(f" 🚀 [数据导出版] 开始网格搜索 & 敏感度分析 | 模型: {strategy_class.__name__}")
    print("=" * 80)

    # 1. 解析传入的多维网格空间
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    param_combinations = [dict(zip(keys, v)) for v in product(*values)]
    
    total_steps = len(param_combinations)
    start_time = time.time()

    # ⚡ 用于存放结构化运行结果的列表（直接扁平化，方便后续画图或转 DataFrame）
    history_records = []
    best_train_acc = 0.0
    best_params = {}

    # 2. 极速 Batch 寻优循环
    for current_step, params in enumerate(param_combinations, 1):
        strategy = strategy_class(**params)
        
        # 并行化矩阵计算准确率
        train_preds = strategy.predict_batch(train_data)
        train_acc = np.mean(train_preds == 1)
        
        # ⚡ 记录每一步的参数与对应的准确率
        record = params.copy()
        record["train_acc"] = float(train_acc)
        history_records.append(record)
        
        if train_acc > best_train_acc:
            best_train_acc = train_acc
            best_params = params

        # 动态终端进度条
        if current_step % 10 == 0 or current_step == total_steps:
            percent = (current_step / total_steps) * 100
            elapsed = time.time() - start_time
            eta = (elapsed / current_step) * (total_steps - current_step)
            eta_str = f"{int(eta)}s" if eta < 60 else f"{int(eta//60)}m{int(eta%60)}s"
            bar = '█' * int(round(20 * percent / 100)) + '-' * (20 - int(round(20 * percent / 100)))
            sys.stdout.write(f"\r 搜索进度: [{bar}] {percent:.1f}% | 最佳Train: {best_train_acc:.2%} | ETA: {eta_str}")
            sys.stdout.flush()

    print("\n" + "-" * 80)
    print(f" 🎯 最佳参数组合: {best_params} | 最佳 Train Acc: {best_train_acc:.2%}")
    
    # 3. 盲测独立测试集
    final_test_strategy = strategy_class(**best_params)
    test_preds = final_test_strategy.predict_batch(test_data)
    final_test_acc = np.mean(test_preds == 1)
    print(f" 🔒 测试集 (Test Data) 终局盲测准确率: {final_test_acc:.2%}")
    print("-" * 80)

    # 4. 📊 计算参数的宏观敏感度
    print(" 📊 [参数影响力战力榜] (基于 Train 数据集波动极差):")
    target_params = [k for k, v in param_grid.items() if len(v) > 1]
    sensitivity_scores = {}

    for p_name in target_params:
        unique_vals = param_grid[p_name]
        max_per_val = []
        for val in unique_vals:
            accs = [rec["train_acc"] for rec in history_records if rec[p_name] == val]
            if accs:
                max_per_val.append(np.mean(accs))
                
        p_range = np.max(max_per_val) - np.min(max_per_val) if len(max_per_val) > 1 else 0.0
        sensitivity_scores[p_name] = p_range

    total_score = sum(sensitivity_scores.values()) if sum(sensitivity_scores.values()) > 0 else 1.0
    sorted_sensitivity = sorted(sensitivity_scores.items(), key=lambda x: x[1], reverse=True)
    
    for rank, (name, score) in enumerate(sorted_sensitivity, 1):
        relative_pct = (score / total_score) * 100
        bar_visual = '🟩' * int(round(relative_pct / 5)) 
        print(f"   Top {rank} | {name:<12} : {bar_visual:<20} {relative_pct:.1f}% (准确率绝对波动贡献: {score:.3%})")
        
    print("=" * 80 + "\n")
    
    # ⚡ 增加 history_records 返回
    return best_params, final_test_acc, history_records

In [20]:
class LogitInteractionStrategy(BasePredictionStrategy):
    def __init__(self, w_base=100.0, alpha=0.01, beta=0.01, min_matches=20):
        """
        :param w_base: 基础对数几率差的放大权重（用于与协同、克制指数对齐量纲）
        :param alpha: 队友协同对数几率的缩放权重
        :param beta: 对手克制对数几率的缩放权重
        :param min_matches: 贝叶斯平滑常数，用于稳定低频组合
        """
        self.w_base = w_base
        self.alpha = alpha
        self.beta = beta
        self.min_matches = min_matches

    def train(self, train_data):
        # 纯公式解析策略，不需要机器学习拟合
        pass

    def _wr_to_logit_batch(self, wr_array):
        """
        ⚡ 矩阵化计算对数几率：Logit = ln(p / (1-p))
        带有 1e-5 的安全截断，防止胜率刚好为 0 或 1 时导致 log(0) 报错
        """
        wr_clipped = np.clip(wr_array, 1e-5, 1.0 - 1e-5)
        return np.log(wr_clipped / (1.0 - wr_clipped))

    def predict_batch(self, samples):
        """
        ⚡ 全矩阵化 Batch 预测，接收样本列表，直接返回 0/1 的 NumPy 数组
        """
        batch_size = len(samples)
        if batch_size == 0:
            return np.array([], dtype=np.int32)

        # 1. 计算个体基础对数几率差并乘以 w_base 放大
        # 提取所有样本的基础胜率 -> 维度均为 [Batch, 5]
        win_base_wr = np.array([s["win_base_wr"] for s in samples], dtype=np.float32)
        lose_base_wr = np.array([s["lose_base_wr"] for s in samples], dtype=np.float32)

        # 矩阵化转换为 logit 空间并在队伍轴(axis=1)上求和 -> [Batch]
        logit_team1_base = self._wr_to_logit_batch(win_base_wr).sum(axis=1)
        logit_team2_base = self._wr_to_logit_batch(lose_base_wr).sum(axis=1)
        
        # ⚡ 核心改动：对大盘基础对数几率差进行 w_base 缩放
        base_diff = (logit_team1_base - logit_team2_base) * self.w_base

        # 2. 调用基类的辅助函数，批量获取平滑后的关系矩阵 -> [Batch, 5, 5]
        win_syn_smooth = self._get_smoothed_vals_batch([s["win_synergy"] for s in samples])
        lose_syn_smooth = self._get_smoothed_vals_batch([s["lose_synergy"] for s in samples])
        cnt_smooth = self._get_smoothed_vals_batch([s["win_vs_lose_counter"] for s in samples])

        # 3. 计算双方队伍内部的协同对数几率差 (消除对角线 i == j)
        eye_mask = np.eye(5, dtype=bool)
        win_syn_smooth[:, eye_mask] = 0.0
        lose_syn_smooth[:, eye_mask] = 0.0

        # 对 5x5 区域进行整体求和 -> [Batch]
        syn_team1 = win_syn_smooth.sum(axis=(1, 2))
        syn_team2 = lose_syn_smooth.sum(axis=(1, 2))
        syn_diff = syn_team1 - syn_team2

        # 4. 计算跨队伍的对位克制对数几率和 -> [Batch]
        counter_diff = cnt_smooth.sum(axis=(1, 2))

        # 5. 线性加权合成最终的对数几率差
        total_delta_logit = base_diff + (self.alpha * syn_diff) + (self.beta * counter_diff)

        # 二分类预测，直接根据正负号判断谁赢 (输出为 1 或 0)
        return np.where(total_delta_logit >= 0, 1, 0)

In [21]:
class DynamicWrLogitStrategy(BasePredictionStrategy):
    def __init__(self, w_base=100.0, alpha=1.0, beta=1.0, min_matches=25):
        """
        :param w_base: 最终两队团队 Logit 战力的全局放大权重
        :param alpha: 队友协同对局内胜率的影响权重（默认1.0表示直接应用原始概率修正）
        :param beta: 对手克制对局内胜率的影响权重（默认1.0表示直接应用原始概率修正）
        :param min_matches: 贝叶斯平滑常数
        """
        self.w_base = w_base
        self.alpha = alpha
        self.beta = beta
        self.min_matches = min_matches

    def train(self, train_data):
        pass

    def predict_batch(self, samples):
        """
        ⚡ 全矩阵化 Batch 预测，接收样本列表，直接返回 0/1 的 NumPy 数组
        """
        batch_size = len(samples)
        if batch_size == 0:
            return np.array([], dtype=np.int32)

        # 1. 提取双方基础胜率 -> 维度均为 [Batch, 5]
        win_base_wr = np.array([s["win_base_wr"] for s in samples], dtype=np.float32)
        lose_base_wr = np.array([s["lose_base_wr"] for s in samples], dtype=np.float32)

        # 2. 提取并平滑 5x5 的关系矩阵 -> 维度均为 [Batch, 5, 5]
        win_syn_smooth = self._get_smoothed_vals_batch([s["win_synergy"] for s in samples])
        lose_syn_smooth = self._get_smoothed_vals_batch([s["lose_synergy"] for s in samples])
        cnt_smooth = self._get_smoothed_vals_batch([s["win_vs_lose_counter"] for s in samples])

        # 3. 队友协同分修正 (消除对角线 i == j)
        eye_mask = np.eye(5, dtype=bool)
        win_syn_smooth[:, eye_mask] = 0.0
        lose_syn_smooth[:, eye_mask] = 0.0

        # 对矩阵按行(axis=2)求和，计算每个英雄分到的协同加成 -> [Batch, 5]
        # 注意：需要除以 self.w_base 将百分比转换为概率空间
        win_syn_bonus = (win_syn_smooth.sum(axis=2) / self.w_base) * self.alpha
        lose_syn_bonus = (lose_syn_smooth.sum(axis=2) / self.w_base) * self.alpha

        # 4. 对手克制分修正
        # 队1(i) 对 队2(j) 的克制：对行(axis=2)求和得到队1每个英雄受到的总克制加成 -> [Batch, 5]
        win_cnt_bonus = (cnt_smooth.sum(axis=2) / self.w_base) * self.beta
        
        # 队2(j) 对 队1(i) 的反向克制：对列(axis=1)求和且取反，得到队2每个英雄受到的总克制加成 -> [Batch, 5]
        lose_cnt_bonus = ((-cnt_smooth).sum(axis=1) / self.w_base) * self.beta

        # 5. 融合得到每个英雄的动态实际胜率 -> [Batch, 5]
        final_win_hero_wr = win_base_wr + win_syn_bonus + win_cnt_bonus
        final_lose_hero_wr = lose_base_wr + lose_syn_bonus + lose_cnt_bonus

        # 6. 调用基类函数，整体投影到 Logit 空间，并在团队轴(axis=1)上累加 -> [Batch]
        logit_team1 = self._wr_to_logit_batch(final_win_hero_wr).sum(axis=1)
        logit_team2 = self._wr_to_logit_batch(final_lose_hero_wr).sum(axis=1)

        return np.where(logit_team1 >= logit_team2, 1, 0)

In [23]:
alpha_log_range = [0.0] + list(np.geomspace(0.001, 100.0, num=20))
beta_log_range  = [0.0] + list(np.geomspace(0.001, 100.0, num=20))
min_matches_range = [0.0] + list(np.geomspace(1, 100.0, num=5))

# 构建你想要的任何网格
custom_grid = {
    "w_base": [100.0],
    "alpha": alpha_log_range,
    "beta": beta_log_range,
    "min_matches": min_matches_range
}

all_res = []
for strategy in [OptimizedSimpleStrategy, LogitInteractionStrategy, DynamicWrLogitStrategy]:
    best_cfg, test_accuracy, res = run_train_search_and_test_batch(
        strategy_class=strategy, # 换成另外两个类也完全通用
        param_grid=custom_grid,
        train_data=train_data,
        test_data=test_data
    )
    all_res.append(res)

 🚀 [数据导出版] 开始网格搜索 & 敏感度分析 | 模型: OptimizedSimpleStrategy
 搜索进度: [--------------------] 2.3% | 最佳Train: 54.71% | ETA: 29m16s

KeyboardInterrupt: 

In [1]:
_evaluate_single_combination(OptimizedSimpleStrategy, {'w_base': 0, 'alpha': 0.07196856730011521, 'beta': 0.08286427728546843, 'min_matches': 0.0}, test_matrix_data)

NameError: name '_evaluate_single_combination' is not defined